In [1]:
import gzip
import os
import random
import pandas as pd
import numpy as np

from collections import defaultdict

os.chdir(r'/Users/yerongliu/Downloads/ML2/Final Project/Data')
# os.chdir(r'/Users/liuyerong/Downloads/ML2')

def readGz(f):
    for l in gzip.open(f):
        yield eval(l)

In [2]:
data = []
for l in readGz("train.json.gz"):
    data.append(l)
    
data = pd.DataFrame(data)

In [3]:
# randomnize data set
data = data.sample(frac = 1)

In [4]:
# make up pairing for purchased records
data["pair"] = data["reviewerID"] + "-" + data["itemID"]

In [5]:
# define the not-purchased data size is 1.2 of the purchased records
unique_items = np.unique(data["itemID"])
unique_reviewers = np.unique(data["reviewerID"])
data_one, data_zero = len(data["pair"]), len(data["pair"]) * 1.2

In [7]:
# generate not-purchased data
pair_zero = []
while len(pair_zero) <= data_zero:
    r = random.choice(unique_reviewers)
    i = random.choice(unique_items)
    if (r + "-" + i) not in data["pair"]:
        pair_zero.append(r + "-" + i)

In [8]:
# compile purchased and not-purchased records together and label with "1" and "0"
full_pair = np.vstack([np.array(data["pair"]).reshape(-1,1),np.array(pair_zero).reshape(-1,1)])
full_pair = pd.DataFrame(full_pair)

full_pair["purchase_status"] = [1] * len(data["pair"]) + [0] * len(pair_zero)
full_pair.columns = (["pair", "purchase_status"])

In [10]:
from sklearn.model_selection import train_test_split
train_pair, test_pair = train_test_split(full_pair, test_size = 0.2)

In [16]:
from scipy.sparse import csr_matrix

num_user = len(data['reviewerID'].unique())
num_item = len(data['itemID'].unique())

def csr_matrix(data):
      
    # Map item ID to indices
    item_mapper = dict(zip(np.unique(data["itemID"]), list(range(num_item))))
      
    # Map indices to IDs
    item_inv_mapper = dict(zip(list(range(num_item)), np.unique(data["itemID"])))
      
    user_index = [user_mapper[i] for i in data['reviewerID']]
    item_index = [item_mapper[i] for i in data['itemID']]
    X = csr_matrix((data["rating"], (item_index, user_index)), shape=(num_item, num_user))
      
    return X, item_mapper, item_inv_mapper 

In [17]:
X, item_mapper, item_inv_mapper = csr_matrix(data)

In [19]:
np.unique(data["rating"])

array([1., 2., 3., 4., 5.])

In [20]:
from sklearn.neighbors import NearestNeighbors

# Find similar items using KNN

def KNN(itemID, X, k, metric='cosine', show_distance=False):
      
    similar_ids = []

    item_idx = item_mapper[itemID]
    item_vec = X[item_idx]
    k += 1
    k_NN = NearestNeighbors(n_neighbors=k, algorithm="brute", metric=metric)
    k_NN.fit(X)
    
    item_vec = item_vec.reshape(1,-1)
    neighbouring = k_NN.kneighbors(item_vec, return_distance=show_distance)
    
    for i in range(0,k):
        n = neighbouring.item(i)
        similar_ids.append(item_inv_mapper[n])
    similar_ids.pop(0)
    
    return similar_ids

In [29]:
item_vec = pd.DataFrame(data.groupby('itemID')['rating'].agg(['mean','count']))
item_vec = item_vec.sort_values(by ='count',ascending=False)
item_vec.reset_index()

,itemID,mean,count
0,I835857013,4.185022,227
1,I964877831,4.257143,210
2,I431429328,4.561224,196
3,I359425229,4.040816,196
4,I408729378,3.982857,175
...,...,...,...
19909,I029808566,5.000000,1
19910,I714987535,5.000000,1
19911,I170252582,3.000000,1
19912,I113106808,4.000000,1


In [242]:
reviewer_vec = pd.DataFrame(data.groupby('reviewerID')['rating'].agg(['mean','count']))
reviewer_vec = reviewer_vec.sort_values(by ='count',ascending=False)

In [38]:
max_ct = max(item_vec["count"])
min_ct = min(item_vec["count"])

In [349]:
import math
k = []
for i in tqdm(np.unique(data['itemID'])):
    k.append(int(15*(math.exp((len(data[data["itemID"] == i]) - min_ct)/(max_ct-min_ct))-1)))

100%|█████████████████████████████████████| 19914/19914 [03:40<00:00, 90.44it/s]


In [405]:
from tqdm import tqdm

rec_mapper = defaultdict(list)

for i in tqdm(np.unique(data['itemID'])):
#     k = int(50*((len(data[data["itemID"] == i]) - min_ct)/(max_ct-min_ct)))
    k= (int(25*(math.exp((len(data[data["itemID"] == i]) - min_ct)/(max_ct-min_ct))-1)))
    # k is 50*popularity index 
    rec_per_item = KNN(i, X, k)
    rec_mapper[i].append(rec_per_item)

100%|█████████████████████████████████████| 19914/19914 [04:42<00:00, 70.38it/s]


In [59]:
train_reviewer = []
train_item = []

for l in train_pair["pair"]:
    r, i = l.strip().split("-")
    train_reviewer.append(r)
    train_item.append(i)

unique_train_reviewer = np.unique(train_reviewer)
unique_train_item = np.unique(train_item)

In [165]:
pair_mapper = defaultdict(list)
item_to_rec_reviwer_mapper = defaultdict(list)

for i in unique_train_item:
    pair_mapper[i]
    item_to_rec_reviwer_mapper[i]

train = zip(train_reviewer, train_item)
for i in train:
    pair_mapper[i[1]].append(i[0])

In [396]:
pred_purchase_list = []

for i in unique_train_item:
    for r in rec_mapper[i]:
        for item in r:
            reviewer_list = pair_mapper[item]
            for u in reviewer_list:
                pred_purchase_list.append(u + "-" + i)

In [341]:
mean_purchase_ct = sum(reviewer_vec["count"])/len(reviewer_vec["count"]) 
unfreq_reviewer = reviewer_vec[reviewer_vec['count'] > (int(mean_purchase_ct)-1)].reset_index()
unfreq_reviewer = np.array(unfreq_reviewer)

In [340]:
popular_vec = item_vec[(item_vec['mean'] > 3) & (item_vec['count'] > 8.5)].reset_index()
popular_vec = np.array(popular_vec)

In [306]:
train_pred_purchase = []
train = zip(train_reviewer, train_item)
for r, i in tqdm(train):
    temp_pair = (r + "-" + i)
    if temp_pair in pred_purchase_list:
        train_pred_purchase.append(1)
    elif i in popular_vec:
        train_pred_purchase.append(1)
    elif r in unfreq_reviewer:
        train_pred_purchase.append(0)
    else: train_pred_purchase.append(0)

352000it [52:41, 111.34it/s]


In [310]:
len(train_pred_purchase), len(train_pair["purchase_status"])

(352000, 352000)

In [311]:
train_accuracy = float(sum([(a == b) for a,b in zip(train_pred_purchase, 
                                             train_pair["purchase_status"])]))/len(train_pred_purchase)
train_accuracy

0.6623238636363636

In [312]:
missed = 0
over_ct = 0
for a, b in zip(train_pred_purchase, train_pair["purchase_status"]):
    if a == 0 and b == 1:
        missed += 1
    if a == 1 and b == 0:
        over_ct += 1
print(missed, over_ct)

62932 55930


In [313]:
test_pred_purchase = []
test = zip(test_reviewer, test_item)
for r, i in tqdm(test):
    temp_pair = (r + "-" + i)
    if temp_pair in pred_purchase_list:
        test_pred_purchase.append(1)
    elif i in popular_vec:
        test_pred_purchase.append(1)
    elif r in unfreq_reviewer:
        test_pred_purchase.append(0)
    else: test_pred_purchase.append(0)
        
test_accuracy = float(sum([(a == b) for a,b in zip(test_pred_purchase, 
                                             test_pair["purchase_status"])]))/len(test_pred_purchase)
test_accuracy

88001it [07:31, 195.12it/s]


0.6582538834785968

In [402]:
mean_purchase_ct = sum(reviewer_vec["count"])/len(reviewer_vec["count"]) 
unfreq_reviewer = reviewer_vec[reviewer_vec['count'] > (int(mean_purchase_ct)-3)].reset_index()
unfreq_reviewer = np.array(unfreq_reviewer)

In [398]:
popular_vec = item_vec[(item_vec['mean'] > 3) & (item_vec['count'] > 9)].reset_index()
popular_vec = np.array(popular_vec)

In [399]:
predictions = open("predictions_Purchase.txt", 'w')
for l in open("pairs_Purchase.txt"):
    predictions.write(l)

pred_Purchase = pd.read_csv('predictions_Purchase.txt') 

In [403]:
pred = []
for l in tqdm(pred_Purchase["reviewerID-itemID"]):
    r, i = l.strip().split("-")
    if l in pred_purchase_list:
        pred.append(1)
    elif i in popular_vec:
        pred.append(1)
    elif r in unfreq_reviewer:
        pred.append(0)
    else: pred.append(0)

100%|████████████████████████████████████| 28000/28000 [04:01<00:00, 115.92it/s]


In [404]:
pred_Purchase["prediction"] = pred
pred_Purchase.to_csv (r'/Users/yerongliu/Downloads/ML2/Final Project/Data/predictions_Purchase.csv', index=None)